In [3]:
import os, shutil
import pandas as pd
from rdkit import Chem
from DFTStructureGenerator.xtb_process import read_xyz
from DFTStructureGenerator import FormatConverter 

In [2]:
import numpy as np

def kabsch(P, Q):
    """
    Compute optimal rotation matrix using Kabsch algorithm
    P, Q: (N,3) coordinates
    """
    C = np.dot(P.T, Q)
    V, S, Wt = np.linalg.svd(C)

    d = np.sign(np.linalg.det(np.dot(V, Wt)))
    D = np.diag([1, 1, d])

    U = np.dot(np.dot(V, D), Wt)
    return U


def align_molecules(coords_list, atom_indices):
    """
    Align a list of molecular coordinates to the first structure.

    Parameters
    ----------
    coords_list : list of np.ndarray
        List of coordinates (N_atoms, 3)
    atom_indices : list[int]
        Indices of atoms used for alignment

    Returns
    -------
    aligned_coords_list : list of np.ndarray
        Aligned coordinates
    """

    ref_coords = coords_list[0].copy()
    aligned = [ref_coords]

    ref_subset = ref_coords[atom_indices]
    ref_centroid = ref_subset.mean(axis=0)
    ref_subset_centered = ref_subset - ref_centroid

    for coords in coords_list[1:]:

        subset = coords[atom_indices]
        centroid = subset.mean(axis=0)

        subset_centered = subset - centroid

        # compute rotation
        R = kabsch(subset_centered, ref_subset_centered)

        # apply transform to whole molecule
        centered = coords - centroid
        rotated = np.dot(centered, R)
        aligned_coords = rotated + ref_centroid

        aligned.append(aligned_coords)

    return aligned

In [15]:
def rotation(a, sin, cos):
    a = np.array(a)
    a = a / np.sqrt(a @ a.T)
    u, v, w = a
    return np.array([[u * u + (1 - u * u) * cos, u * v * (1 - cos) - w * sin, u * w * (1 - cos) + v * sin, 0],
                     [u * v * (1 - cos) + w * sin, v * v + (1 - v * v)
                      * cos, v * w * (1 - cos) - u * sin, 0],
                     [u * w * (1 - cos) - v * sin, v * w * (1 - cos) +
                      u * sin, w * w + (1 - w * w) * cos, 0],
                     [0, 0, 0, 1]])

def rot_mol(position, axis=np.array([0, 1, 0]), sin=0, cos=-1):
    # Define rotation corner is 180 degree
    up_position = np.insert(position, 3, np.ones(len(position)), 1).T
    matrix = rotation(axis, sin, cos)
    up_position = (matrix @ up_position).T
    return up_position[:, :3]

In [31]:
atoms, positions = read_xyz(r'D:\OneDrive_all\OneDrive\Work\文章\6_Borane_Dataset\CView\B_00388_Nu_00137_r.xyz')
first_position = rot_mol(positions[0], axis=np.array([1, 0, 0]), sin=1, cos=0)
first_position = rot_mol(first_position, axis=np.array([0, -1, 0]), sin=1, cos=0)
positions[0] = first_position
new_positions = align_molecules(np.array(positions), [0, 8, 7])
FormatConverter.mol_to_xyz(None, atoms[0], new_positions, r'E:\work\VMD\B_00388_Nu_00137_r.xyz')
# new_mol = [{"name": idx, 'atomlist': atom, 'positions': pos} for idx, [atom, pos] in enumerate(zip(atoms, new_positions))]
# FormatConverter.write_xyz_file(r'D:\OneDrive_all\OneDrive\Work\文章\6_Borane_Dataset\CView\B_00388_Nu_00137_Cl_00504_aligned.xyz', new_mol)

['E:\\work\\VMD\\B_00388_Nu_00137_r_0.xyz',
 'E:\\work\\VMD\\B_00388_Nu_00137_r_1.xyz',
 'E:\\work\\VMD\\B_00388_Nu_00137_r_2.xyz',
 'E:\\work\\VMD\\B_00388_Nu_00137_r_3.xyz',
 'E:\\work\\VMD\\B_00388_Nu_00137_r_4.xyz']